In [ ]:
#@title **Librerías y Datos**
!pip install kagglehub
!pip install timm
!pip install torchmetrics


import os
import librosa
import numpy as np
import pandas as pd
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import kagglehub
import random

from pathlib import Path
from torchmetrics import Accuracy
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)



os.environ['KAGGLE_API_TOKEN'] = 'KGAT_f6736a1438329933cf2af4cb089b7c12'
path = kagglehub.competition_download('birdclef-2025')
print(path)

os.listdir(path)


SEED = 42
random.seed(SEED)
np.random.seed(SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.9 MB/s eta 0:00:00


100%|██████████| 11.5G/11.5G [10:59<00:00, 18.7MB/s]

Extracting files...


/root/.cache/kagglehub/competitions/birdclef-2025


In [ ]:
#title **CONFIGURACIÓN**
DATASET_PATH = path

AUDIO_FOLDER = os.path.join(DATASET_PATH, "train_audio")

OUTPUT_DIR = "dataset_npy"

# Audio
SR = 32000

# Duración por segmento
SEGMENT_DURATION = 5

# Solapamiento entre ventanas
OVERLAP = 0.5

# Parámetros espectrograma
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

# Split
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15

#Directorios
for split in ["train", "val", "test"]:

    os.makedirs(
        os.path.join(OUTPUT_DIR, split),
        exist_ok=True
    )

In [ ]:
#title **OBTENER AUDIOS**
audio_files = []

for root, dirs, files in os.walk(AUDIO_FOLDER):

    for file in files:

        if file.endswith((".ogg", ".wav", ".mp3")):

            full_path = os.path.join(root, file)

            species = Path(root).name

            audio_files.append({
                "path": full_path,
                "species": species
            })

df = pd.DataFrame(audio_files)

print("Total audios:", len(df))


Total audios: 28564


In [ ]:
#title **DIVISIÓN DE DATOS**

# First split: 70% train, 30% temp (for validation + test)
train_df, temp_df = train_test_split(
    df,
    test_size=(1 - TRAIN_SIZE),
    stratify=df["species"],
    random_state=SEED
)

# Examine temp_df for single-instance species before the second split
label_counts_temp = temp_df['species'].value_counts()
single_instance_species_temp = label_counts_temp[label_counts_temp < 2].index

# Separate temp_df into a stratifiable part and a non-stratifiable part
temp_df_stratifiable = temp_df[~temp_df['species'].isin(single_instance_species_temp)]
temp_df_non_stratifiable = temp_df[temp_df['species'].isin(single_instance_species_temp)]

print(f"\nFilas en temp_df que no pueden ser estratificadas (clases de una sola instancia): {len(temp_df_non_stratifiable)}")
print(f"Número de clases únicas en temp_df antes de la segunda estratificación: {len(label_counts_temp)}")
print(f"Número de clases únicas que pueden ser estratificadas en temp_df: {len(temp_df_stratifiable['species'].unique())}")

# Second split: 1/3 validation, 2/3 test from the stratifiable part of temp_df
# This corresponds to approximately VAL_SIZE and TEST_SIZE of the original df
validation_df_stratified, test_df_stratified = train_test_split(
    temp_df_stratifiable,
    test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE), # Calculate proportion for the second split
    stratify=temp_df_stratifiable["species"],
    random_state=SEED
)

# Combine the stratified results with the non-stratifiable rows
# Non-stratifiable rows are added to the test set to ensure they are not lost.
val_df = validation_df_stratified
test_df = pd.concat([test_df_stratified, temp_df_non_stratifiable])

print("\nTrain:", len(train_df), f"({len(train_df)/len(df):.2%})")
print("Val:", len(val_df), f"({len(val_df)/len(df):.2%})")
print("Test:", len(test_df), f"({len(test_df)/len(df):.2%})")


Filas en temp_df que no pueden ser estratificadas (clases de una sola instancia): 30
Número de clases únicas en temp_df antes de la segunda estratificación: 206
Número de clases únicas que pueden ser estratificadas en temp_df: 176

Train: 19994 (70.00%)
Val: 4270 (14.95%)
Test: 4300 (15.05%)


In [ ]:
#@title **Funciones para Datos**

# FUNCIONES AUDIO
def load_audio(file_path):

    y, sr = librosa.load(
        file_path,
        sr=SR
    )

    return y

# SEGMENTACIÓN
def split_audio(y):

    segment_samples = int(SR * SEGMENT_DURATION)

    step = int(segment_samples * (1 - OVERLAP))

    segments = []

    for start in range(0, len(y), step):

        end = start + segment_samples

        segment = y[start:end]

        if len(segment) < segment_samples:

            padding = segment_samples - len(segment)

            segment = np.pad(segment, (0, padding))

        segments.append(segment)

        if end >= len(y):
            break

    return segments

# DATA AUGMENTATION
def augment_audio(y):

    aug_type = random.choice([
        "none",
        "noise",
        "gain",
        "pitch"
    ])


    # Ruido suave
    if aug_type == "noise":

        noise = np.random.normal(
            0,
            0.003,
            len(y)
        )

        y = y + noise


    # Cambio de pitch
    elif aug_type  == "pitch":

        steps = random.uniform(-1, 1)

        y = librosa.effects.pitch_shift(
            y,
            sr=SR,
            n_steps=steps
        )


    # Cambio de volumen
    elif aug_type  == "gain":

        gain = random.uniform(0.8, 1.2)

        y = y * gain

    return y

# ESPECTOGRAMA
def create_mel(segment):

    mel = librosa.feature.melspectrogram(
        y=segment,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    return mel_db.astype(np.float32)


#RANDOM CROP
def random_crop(y):

    target_samples = int(
        SEGMENT_DURATION * SR
    )

    # SI ES CORTO → PADDING
    if len(y) < target_samples:

        padding = target_samples - len(y)

        y = np.pad(
            y,
            (0, padding)
        )

        return y

    # SMART SAMPLING
    max_attempts = 10

    for _ in range(max_attempts):

        start = random.randint(
            0,
            len(y) - target_samples
        )

        segment = y[
            start:start + target_samples
        ]


        # ENERGÍA DEL SEGMENTO
        energy = np.mean(
            np.abs(segment)
        )

        # evitar silencios
        if energy > 0.01:

            return segment

    # fallback
    return segment

    #ESPECTOGRAMA
def create_mel(y):

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    mel = librosa.power_to_db(
        mel,
        ref=np.max
    )

    # normalización
    mel = (mel - mel.mean()) / (
        mel.std() + 1e-6
    )

    return mel.astype(np.float32)

#CREAR DATAFRAME
def build_dataframe(audio_root):

    files = []

    for root, dirs, filenames in os.walk(audio_root):

        for file in filenames:

            if file.endswith((
                ".ogg",
                ".wav",
                ".mp3"
            )):

                species = Path(root).name

                files.append({
                    "path": os.path.join(root, file),
                    "species": species
                })

    return pd.DataFrame(files)

In [ ]:
#@title **DATASET**

class BioacousticDataset(Dataset):

    def __init__(
        self,
        dataframe,
        species_to_idx,
        taxonomic_map,
        training=True
    ):

        self.df = dataframe

        self.species_to_idx = species_to_idx

        self.taxonomic_map = taxonomic_map

        self.training = training

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        audio_path = row["path"]

        species_name = row["species"]

        # CARGAR AUDIO
        y, _ = librosa.load(
            audio_path,
            sr=SR
        )

        # SEGMENTACIÓN ONLINE
        y = random_crop(y)

        # AUGMENTATION SOLO TRAIN
        if self.training:

            y = augment_audio(y)

        # MEL SPECTROGRAM
        mel = create_mel(y)

        # TENSOR
        mel = torch.tensor(
            mel,
            dtype=torch.float32
        )

        # [1, H, W]
        mel = mel.unsqueeze(0)

        # LABELS
        species_label = self.species_to_idx[
            species_name
        ]

        taxonomic_label = self.taxonomic_map[
            species_name
        ]

        return (
            mel,
            torch.tensor(species_label),
            torch.tensor(taxonomic_label)
        )

In [ ]:
#@title **DATAFRAME**

print(df.head())

                                                path  species
0  /root/.cache/kagglehub/competitions/birdclef-2...  crbtan1
1  /root/.cache/kagglehub/competitions/birdclef-2...  crbtan1
2  /root/.cache/kagglehub/competitions/birdclef-2...  crbtan1
3  /root/.cache/kagglehub/competitions/birdclef-2...  crbtan1
4  /root/.cache/kagglehub/competitions/birdclef-2...  crbtan1


In [ ]:
#@title **LABELS**
species_list = sorted(
    df["species"].unique()
)

species_to_idx = {
    s: i
    for i, s in enumerate(species_list)
}

NUM_SPECIES = len(species_list)

In [ ]:
#@title **MAPA TAXONOMICO**
# Assuming 4 taxonomic classes as used in BioacousticModel (num_taxonomic=4)
NUM_TAXONOMIC_CLASSES = 4

taxonomic_map = {}
for species_name, idx in species_to_idx.items():
    # Assign each species to one of the 4 taxonomic classes for demonstration
    # If actual taxonomic data is available, it should be loaded and used here.
    taxonomic_map[species_name] = idx % NUM_TAXONOMIC_CLASSES

In [ ]:
#@title **DATASETS**
train_dataset = BioacousticDataset(
    train_df,
    species_to_idx,
    taxonomic_map,
    training=True
)

val_dataset = BioacousticDataset(
    val_df,
    species_to_idx,
    taxonomic_map,
    training=False
)

test_dataset = BioacousticDataset(
    test_df,
    species_to_idx,
    taxonomic_map,
    training=False
)

In [ ]:
#@title **BALANCED SAMPLER**
species_counts = train_df[
    "species"
].value_counts()

sample_weights = []

for species in train_df["species"]:

    weight = 1.0 / species_counts[species]

    sample_weights.append(weight)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

In [ ]:
#@title **DATA LOADERS**
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
#@title **TEST**
for mel, species, taxonomic in train_loader:

    print("Mel:", mel.shape)

    print("Species:", species.shape)

    print("Taxonomic:", taxonomic.shape)

    break

Mel: torch.Size([16, 1, 128, 313])
Species: torch.Size([16])
Taxonomic: torch.Size([16])


In [ ]:
#@title **SPEC AUGMENT**

class SpecAugment(nn.Module):

    def __init__(
        self,
        freq_mask=10,
        time_mask=20
    ):

        super().__init__()

        self.freq_mask = freq_mask
        self.time_mask = time_mask

    def forward(self, x):

        B, C, F, T = x.shape

        # Frequency mask
        f = np.random.randint(0, self.freq_mask)

        f0 = np.random.randint(0, F - f)

        x[:, :, f0:f0+f, :] = 0

        # Time mask
        t = np.random.randint(0, self.time_mask)

        t0 = np.random.randint(0, T - t)

        x[:, :, :, t0:t0+t] = 0

        return x


In [ ]:
#@title **CONFORMER BLOCK**

class ConformerBlock(nn.Module):

    def __init__(
        self,
        dim,
        num_heads=4
    ):

        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            batch_first=True
        )

        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

        self.norm1 = nn.LayerNorm(dim)

        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):

        attn_out, _ = self.attn(
            x,
            x,
            x
        )

        x = self.norm1(x + attn_out)

        ff_out = self.ffn(x)

        x = self.norm2(x + ff_out)

        return x


In [ ]:
#@title **ATTENTION POOLING**

class AttentionPooling(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(dim, dim),
            nn.Tanh(),
            nn.Linear(dim, 1)
        )

    def forward(self, x):

        weights = self.attention(x)

        weights = torch.softmax(
            weights,
            dim=1
        )

        pooled = (x * weights).sum(dim=1)

        return pooled


In [ ]:
#@title **MODELO COMPLETO**

class BioacousticModel(nn.Module):

    def __init__(
        self,
        num_species,
        num_taxonomic
    ):

        super().__init__()


        # SpecAugment
        self.specaug = SpecAugment()

        # EfficientNet
        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            in_chans=1,
            num_classes=0
        )

        feature_dim = self.backbone.num_features

         # CONFORMER
        self.conformer = nn.Sequential(

            ConformerBlock(feature_dim),

            ConformerBlock(feature_dim)
        )

        # ATTENTION POOLING
        self.pooling = AttentionPooling(
            feature_dim
        )

       # EMBEDDING
        self.embedding = nn.Sequential(

            nn.Linear(feature_dim, 512),

            nn.ReLU(),

            nn.Dropout(0.3)
        )

        # HEADS
        self.taxonomic_head = nn.Linear(
            512,
            num_taxonomic
        )

        self.species_head = nn.Linear(
            512,
            num_species
        )

    def forward(self, x):

        # AUGMENTATION
        if self.training:

            x = self.specaug(x)


        # CNN
        feat = self.backbone.forward_features(x)

        # feat:
        # [B, C, H, W]

        B, C, H, W = feat.shape

        # TEMPORAL RESHAPE
        feat = feat.mean(dim=2)

        feat = feat.permute(0, 2, 1)

        # [B, T, C]

        # CONFORMER
        feat = self.conformer(feat)

        # POOLING
        pooled = self.pooling(feat)


        # EMBEDDING
        emb = self.embedding(pooled)


        # HEADS
        tax_logits = self.taxonomic_head(emb)

        species_logits = self.species_head(emb)

        return (
            tax_logits,
            species_logits
        )


In [ ]:
#@title **CLASS BALANCED FOCAL LOSS**

class CBFocalLoss(nn.Module):

    def __init__(
        self,
        samples_per_class,
        beta=0.9999,
        gamma=2
    ):

        super().__init__()

        effective_num = 1.0 - np.power(
            beta,
            samples_per_class
        )

        weights = (1.0 - beta) / effective_num

        weights = weights / np.sum(weights)

        self.weights = torch.tensor(
            weights,
            dtype=torch.float32
        )

        self.gamma = gamma

    def forward(self, logits, targets):

        weights = self.weights.to(
            logits.device
        )

        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction='none',
            weight=weights
        )

        pt = torch.exp(-ce_loss)

        focal_loss = (
            (1 - pt) ** self.gamma
        ) * ce_loss

        return focal_loss.mean()


In [ ]:
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = BioacousticModel(
    num_species=NUM_SPECIES,
    num_taxonomic=4
)

model = model.to(device)

# LOSSES

species_loss_fn = CBFocalLoss(
    samples_per_class=species_counts
)

taxonomic_loss_fn = nn.CrossEntropyLoss()


# OPTIMIZER

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

# TRAIN LOOP

for epoch in range(50):

    model.train()

    total_loss = 0

    for mel, species, taxonomic in train_loader:

        mel = mel.to(device)

        species = species.to(device)

        taxonomic = taxonomic.to(device)

        optimizer.zero_grad()

        tax_logits, species_logits = model(mel)

        species_loss = species_loss_fn(
            species_logits,
            species
        )

        tax_loss = taxonomic_loss_fn(
            tax_logits,
            taxonomic
        )

        loss = species_loss + 0.3 * tax_loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1} | Loss: {total_loss:.4f}"
    )


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

/tmp/ipykernel_412/1269480312.py:23: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.weights = torch.tensor(


Epoch 1 | Loss: 480.3174


In [ ]:
#@title **Definir las métricas de precisión**

species_accuracy = Accuracy(task='multiclass', num_classes=NUM_SPECIES).to(device)
taxonomic_accuracy = Accuracy(task='multiclass', num_classes=4).to(device) # Assuming 4 taxonomic classes as used in model init

def evaluate_model(model, data_loader, species_acc_metric, taxonomic_acc_metric, device):
    model.eval()
    species_acc_metric.reset()
    taxonomic_acc_metric.reset()

    with torch.no_grad():
        for mel, species_labels, taxonomic_labels in data_loader:
            mel = mel.to(device)
            species_labels = species_labels.to(device)
            taxonomic_labels = taxonomic_labels.to(device)

            tax_logits, species_logits = model(mel)

            # Calcular precisión de especies
            species_preds = torch.argmax(species_logits, dim=1)
            species_acc_metric.update(species_preds, species_labels)

            # Calcular precisión taxonómica
            tax_preds = torch.argmax(tax_logits, dim=1)
            taxonomic_acc_metric.update(tax_preds, taxonomic_labels)

    species_acc = species_acc_metric.compute()
    taxonomic_acc = taxonomic_acc_metric.compute()

    return species_acc, taxonomic_acc


### Evaluación en el conjunto de Validación

In [ ]:
#@title **Evaluación en el conjunto de validación**
val_species_acc, val_taxonomic_acc = evaluate_model(model, val_loader, species_accuracy, taxonomic_accuracy, device)
print(f"Validation Species Accuracy: {val_species_acc:.4f}")
print(f"Validation Taxonomic Accuracy: {val_taxonomic_acc:.4f}")


### Evaluación en el conjunto de Prueba

In [ ]:
#@title **Evaluación en el conjunto de prueba**
test_species_acc, test_taxonomic_acc = evaluate_model(model, test_loader, species_accuracy, taxonomic_accuracy, device)
print(f"Test Species Accuracy: {test_species_acc:.4f}")
print(f"Test Taxonomic Accuracy: {test_taxonomic_acc:.4f}")


---